# Prompt-only FLUX.2 anchors → one-round SAGE transition video

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MNoichl/FluxFlowMorph/blob/main/notebooks/StillLife_SAGE_Transition_Video.ipynb)

This experimental notebook keeps the established editable anchor-prompt,
RIJKSOIL LoRA, weak blurred/grained continuity, numbered Drive run,
resumability, and cyclic-video setup. It replaces FlowMorph with
**SAGE: Structure-Aware Generative Video Transitions** for one round.
It is pinned to the [v2 paper](https://arxiv.org/html/2510.24667v2)
and the authors' [official code](https://github.com/kan32501/SAGE).

For each circular anchor gap, it:

1. finds foreground line structures with GlueStick;
2. matches them in mask-normalized coordinates with Hungarian assignment;
3. propagates them along a smooth cubic global trajectory while locally
   interpolating each matched segment;
4. rasterizes one structural condition per time step; and
5. uses each condition, both endpoint paintings, interpolated endpoint
   prompt embeddings, and a smooth previous-frame reference to render the
   inbetweens with **the same FLUX.2 Klein 9B + RIJKSOIL LoRA pipeline**.

Important adaptation: the paper starts from **moving clips** and obtains
endpoint motion with SEA-RAFT. These inputs are still paintings, so real
clip flow does not exist. The notebook therefore uses a deterministic,
exposed synthetic-flow bend for the global spline. Everything else above
remains the SAGE structural pipeline. FLUX replaces the paper's FCVG/SVD
renderer because no FLUX.2 Klein ControlNet exists. Exact anchors are
retained between gaps and the last gap returns to the first anchor.


## 1. Editable anchor-generation and SAGE settings

All visible images—including the SAGE-structured inbetweens—use the fused
FLUX.2 Klein + RIJKSOIL model loaded below.


In [ ]:
PROJECT_ROOT = "/content/FlowMorphKlein9B"
REPOSITORY_URL = "https://github.com/MNoichl/FluxFlowMorph.git"
UPDATE_REPOSITORY = True
PROJECT_NAME = "science_path_sage_transition"
LOCAL_ASSET_ROOT = "/content/sage_transition_art"
HF_CACHE_DIR = "/content/hf_cache"

# Drive persistence.
MOUNT_DRIVE = True
DRIVE_PROJECT_BASE = "/content/drive/MyDrive/FluxFlowMorphArt"
RESUME_RUN_DIRECTORY = None

# Editable anchor selection.
BASE_PROMPT_COUNT = None  # None uses every BASE_STAGES entry; any 3..len(BASE_STAGES) works.
REGENERATE_BASE_FRAMES = True

# FLUX.2 Klein Base 9B + RIJKSOIL LoRA.
MODEL_ID = "Runware/BFL-FLUX.2-klein-base-9B"
MODEL_REVISION = "52d7274119d8a2b67f4fba1a43694d9169a44851"
LORA_SOURCE = "MaxNoichl/RIJKSOIL_FLUX2_KLEIN9B_lora_01_000001650"
LORA_REVISION = "042a31d6cd09bf55195f820461fac60b1a358409"
LORA_WEIGHT_NAME = "RIJKSOIL_FLUX2_KLEIN9B_lora_01_000001650.safetensors"
LORA_ADAPTER_NAME = "rijks_oil"
LORA_TRIGGER = "RIJKSOIL"

IMAGE_WIDTH = 1024
IMAGE_HEIGHT = 1024
IMAGE_INFERENCE_STEPS = 50
IMAGE_GUIDANCE_SCALE = 7.0
IMAGE_LORA_SCALE = 1.2
BASE_SEED = 42

# Weak anchor continuity: blurred/grained previous image, no beige canvas.
BASE_CONTINUITY_ENABLED = True
BASE_REFERENCE_BLUR = 16.0
BASE_REFERENCE_GRAIN_STRENGTH = 0.035
BASE_REFERENCE_DENOISE_STRENGTH = 0.75
SAVE_SOFT_REFERENCES = True
FLUX_PROMPT_MAX_SEQUENCE_LENGTH = 512

# Optional cheap anchor trial.
RUN_TRIAL_KEYFRAME = True
TRIAL_KEYFRAME_INDEX = None
TRIAL_SEED = None
TRIAL_DISPLAY_MAX_WIDTH = 768
CONTACT_SHEET_COLUMNS = 6
CONTACT_SHEET_DISPLAY_MAX_WIDTH = 1100

# Automatic foreground masks. Replace individual generated PNGs and rerun
# from section 9 if GrabCut misses the composition.
SAGE_MASK_MODE = "grabcut"  # "grabcut", "full_frame", or "directory"
SAGE_MASK_SOURCE_DIRECTORY = None  # For directory mode: one <anchor_uid>.png per anchor.
SAGE_MASK_REGENERATE = True
SAGE_GRABCUT_MARGIN_FRACTION = 0.035
SAGE_MASK_DILATION_PIXELS = 7
SAGE_MASK_MIN_COVERAGE = 0.04
SAGE_MASK_MAX_COVERAGE = 0.96

# Pinned SAGE structural implementation and GlueStick component.
SAGE_REPOSITORY_URL = "https://github.com/kan32501/SAGE.git"
SAGE_REPOSITORY_COMMIT = "5a30e6bfb035e2c243d90d4804ebda2addecacf4"
SAGE_REPOSITORY_DIRECTORY = "/content/SAGE"
SAGE_GLUESTICK_URL = "https://github.com/cvg/GlueStick/releases/download/v0.1_arxiv/checkpoint_GlueStick_MD.tar"

# One SAGE round per cyclic anchor gap.
SAGE_WIDTH = IMAGE_WIDTH
SAGE_HEIGHT = IMAGE_HEIGHT
SAGE_GENERATED_FRAMES_PER_GAP = 13
SAGE_MAX_POINTS = 1000
SAGE_MAX_LINES = 200
SAGE_MAX_MATCHED_LINES = 160
SAGE_MINIMUM_MATCHED_LINES = 8
SAGE_CONDITION_LINE_WIDTH = 2

# Still-image substitute for unavailable clip flow. 0 gives a direct
# center path; small values curve the global foreground trajectory.
SAGE_SYNTHETIC_FLOW_SCALE = 0.16
SAGE_TRAJECTORY_BEND = 0.04

# FLUX-native rendering of the SAGE guides. The guide is injected into a
# true img2img initialization; both endpoints and that initialization are
# also supplied as FLUX.2 reference images.
SAGE_FLUX_INFERENCE_STEPS = IMAGE_INFERENCE_STEPS
SAGE_FLUX_GUIDANCE_SCALE = IMAGE_GUIDANCE_SCALE
SAGE_FLUX_IMG2IMG_STRENGTH = 0.72
SAGE_ENDPOINT_PALETTE_BLUR = 64.0
SAGE_PREVIOUS_FRAME_BLEND = 0.24
SAGE_PREVIOUS_FRAME_BLUR = 10.0
SAGE_STRUCTURE_INIT_STRENGTH = 0.24
SAGE_STRUCTURE_DILATION_PIXELS = 3
SAGE_INIT_GRAIN_STRENGTH = 0.025
SAGE_REUSE_PROMPT_EMBEDDINGS = True

# Output, reuse, and display.
SAGE_REUSE_COMPLETED_GAPS = True
SAGE_OUTPUT_FPS = 12.0
SAGE_VIDEO_CRF = 16
SAGE_DISPLAY_EACH_GAP_VIDEO = False
SAGE_VIDEO_DISPLAY_WIDTH = 768
RUN_SAGE_RENDER = True  # Set False to stop after mask/line-guide inspection.
DOWNLOAD_FINAL_VIDEO = False


## 2. Editable anchor sciences and prompts

These are copied from the current prompt-only notebook. Edit them here in
exactly the same way; each prompt must contain `RIJKSOIL` exactly once.


In [ ]:
BASE_STAGES = [
    {
        "id": "01_astronomy",
        "science": "Astronomy & Astrophysics",
        "prompt": "RIJKSOIL, a hushed Baroque still life of the heavens brought indoors: a tarnished brass armillary sphere and a celestial globe painted with constellations, an astrolabe and a small orrery, bronze dividers resting on a curling star chart, a pitted meteorite and a shard of quartz catching the light. A single candle stands in for a distant sun on black velvet strewn with faint points of starlight. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, a cold indigo-black palette shot with silver starlight and old brass, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "02_physics",
        "science": "Nuclear, High-Energy, Atomic & Optical Physics",
        "prompt": "RIJKSOIL, a Baroque still life of matter and light: a goblet of uranium glass fluorescing eerie green beside a lead casket cracked to show a faintly glowing vial, a gold-leaf electroscope and a brass tuning fork, a glass prism splitting the candle's beam into a spectral ribbon across polished lenses, and a cloud chamber where fine spiral tracks hang like frozen lightning. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, black and lead-grey lit by uranium-green fluorescence, a prismatic rainbow, and one gold spark, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "03_chemistry_materials",
        "science": "Chemistry & Materials (Organic, Analytical, Materials, Polymers)",
        "prompt": "RIJKSOIL, an alchemical Baroque still life of transformation: a glass alembic and a pear-shaped retort of jewel-coloured liquids, a coiled condenser and a rack of test tubes glowing ruby and cobalt, a burner flame licked green and copper by unseen salts, a brass ball-and-stick molecule, a cluster of iridescent bismuth crystals, a coil of amber resin with an insect trapped inside, and a stone mortar and pestle. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, deep amber and ruby glass with copper-flame green and an iridescent metallic sheen, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "04_geosciences",
        "science": "Geosciences (Water, Atmosphere, Geophysics, Planetary)",
        "prompt": "RIJKSOIL, a Baroque still life of earth and sky: banded agates and mineral specimens, a brass barometer and a glass of layered water and sediment, a small seismograph drum trailing a jagged line, a fossil-bearing rock, and a terrestrial globe half wrapped in drifting mist. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, cool mineral aqua, slate-blue, and misty green, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "05_ecology_evolution",
        "science": "Ecology & Evolution",
        "prompt": "RIJKSOIL, a Baroque still life of the living web and deep time: a bird's nest of speckled eggs among ferns and lichened bark, iridescent beetles and a poised butterfly, a spiral ammonite and a ridged trilobite half-freed from a broken slab of grey limestone with their coils still embedded in the stone, a fern frond pressed as a dark imprint in split shale, a branching red coral for the tree of life, a single weathered skull set back in shadow, and an open naturalist's notebook of careful pencil studies. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, wet forest greens and moss shading into fossil grey-green and bone-ochre, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "06_botany",
        "science": "Botany, Plant & Food Science",
        "prompt": "RIJKSOIL, an opulent Baroque flower and harvest still life in the manner of Rachel Ruysch: tumbling tulips, roses, and poppies just past their prime, an herbarium sheet with a pinned specimen, a magnifier over a veined leaf, split figs and a broken pomegranate, a sheaf of wheat, and a dark loaf beside a comb of honey. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, verdant leaf-green with ripe fruit-reds and gold, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "07_genetics_cell",
        "science": "Genetics, Cell & Molecular Biology",
        "prompt": "RIJKSOIL, a Baroque still life where heredity meets the cell: a spiralling pea tendril twisting like a double helix and open pods with sorted green and yellow peas, beside a pomegranate split to packed glistening arils, a fig cut to its seeded interior, a heaped cluster of translucent gooseberries and glossy fish roe, a comb of honey with rows of hexagonal chambers, and an antique brass microscope whose lens throws a bright disc crowded with round cells caught mid-division. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, pale pearl, milky rose, and soft gold, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "08_medicine_disease",
        "science": "Medicine: Disease, Immunity & Remedy (Immunology, Pharmacology, Oncology)",
        "prompt": "RIJKSOIL, a grave Baroque still life of contagion, remedy, and blood: a pierced silver pomander of dried herbs against the miasma, a curl of bitter cinchona bark, sprigs of rue and rosemary bound with twine, labelled apothecary jars of poppy and foxglove, a hand-blown phial sealed with dark wax, a brass-and-ivory bloodletting fleam, a stoppered flask of dark crimson beside a pale crab laid on cold stone for the old name of the disease, and an old beaked plague-doctor's mask of cracked dark leather, its glass eyes clouded, quiet in the shadow to one side. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, tarnished silver and apothecary amber shading toward blood-crimson, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "09_anatomy_physiology",
        "science": "Anatomy, Surgery, Cardiology & Physiology",
        "prompt": "RIJKSOIL, a solemn Baroque anatomical still life: a small écorché figure and a wax model of the human heart, a gleaming scalpel, forceps, and bone-saw, a Vesalian atlas open to an engraved plate, an hourglass with sand mid-fall and a coiled glass tube, a comb of honey dripping slow for the blood's sweetness, a skull, and a translucent plate glowing faintly like an early radiograph. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, carmine flesh, ivory bone, and cold steel cooling toward a pale radiograph blue, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "10_neuroscience_mind",
        "science": "Neuroscience, Psychiatry & Psychology",
        "prompt": "RIJKSOIL, a Baroque still life of the thinking organ and the interior mind: a human brain suspended in a bell-jar of clear spirit, branching coral and bare winter twigs echoing dendrites, a phrenology bust incised with regions, a faint electric spark leaping a gap, a clouded mirror holding a half-lit face, two theatrical masks of comedy and grief, a slow pendulum, and a single inkblot bleeding on parchment. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, electric blue-violet and shadowed indigo with mirror-silver, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "11_philosophy_society",
        "science": "Philosophy & the Social Sciences",
        "prompt": "RIJKSOIL, a Baroque vanitas of thought and society: a skull resting on a stack of worn leather books, a snuffed candle trailing smoke, an hourglass and a quill in its inkwell, five Platonic solids in glass, an owl in shadow, and beside them brass scales of justice weighing gold coins against a folded contract, an abacus and an open ledger, ivory dice and playing cards for the games of strategy, and a globe half-turned to the dark. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, warm sepia, candle-gold, and coin-gold, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
    {
        "id": "12_computation_math",
        "science": "Computer Science, AI & Mathematics",
        "prompt": "RIJKSOIL, a Baroque still life of pure form and mechanism: an abacus and brass dividers over Euclid's open geometry, interlocking clockwork gears, a chessboard caught mid-game, nested Platonic solids and a small orrery, a perforated brass plate like a punched card, and a single lens turned outward. The candle gutters low, its light circling back toward the stars where the journey began. Painted as a Dutch Golden Age pronkstilleven, the candle the only light, deep chiaroscuro over dark velvet and worn stone, cool brass and silver on black turning toward cosmic indigo, meticulous oil on panel, glinting glass and tarnished brass, a quiet vanitas hush.",
    },
]
 




# [
#     {
#         "id": "nuclear_atomic_optical_physics",
#         "science": "nuclear and high-energy physics; atomic and molecular physics; optics",
#         "prompt": "RIJKSOIL, a medium-wide low three-quarter Dutch Baroque still life rising diagonally from a black laboratory plinth into a stone alcove; a brass cloud chamber beneath a misted glass bell with pale particle tracks; a dark ore specimen in a dull lead cradle; a cut-glass prism catching a narrow muted spectrum; paired brass lenses, a sealed vapor ampoule, an ivory counter dial and a loose arc of copper detector wire; cold upper-left light answered by a low amber glow, pronounced tenebrism, soot black, lead gray, oxidized brass, luminous glass, layered oil glazes and restrained impasto; no people, no readable text.",
#     },
#     {
#         "id": "electronic_magnetic_materials",
#         "science": "electronic, optical and magnetic materials; materials chemistry",
#         "prompt": "RIJKSOIL, a medium-wide lateral Baroque arrangement of broad concentric arcs on a polished slate shelf; a cobalt silicon wafer tilted against a low brass rest; an enamelled copper coil encircling a dark horseshoe magnet; translucent calcite balanced by stepped ferrite tiles; a short fiber-optic strand releasing a few pale points; dark teal silk falling in monumental folds, cool light gathering into warm copper reflections, sculptural chiaroscuro, mineral surfaces, broad brushwork and glazed highlights; no people, no readable text.",
#     },
#     {
#         "id": "mechanics_ocean_aerospace_control",
#         "science": "mechanics and computational mechanics; ocean engineering; aerospace, electrical and control systems engineering",
#         "prompt": "RIJKSOIL, a medium-wide Baroque workshop composition swept by a wing-shaped diagonal above a shallow pewter basin; a brass gyroscope inside its circular gimbal, a small airfoil raised on pins, a steel gear crossed by calipers, a copper-wound servo coupled to a feedback pendulum, and a carved wave crest beside a rolled salt-stained chart; storm-blue canvas and charcoal wool form large shadowed planes; hard left light traces rivets, wet pewter, scratched steel and oil-dark brass with vigorous loaded brushwork; no people, no readable text.",
#     },
#     {
#         "id": "manufacturing_networks",
#         "science": "industrial and manufacturing engineering; computer networks and communications",
#         "prompt": "RIJKSOIL, a medium-wide low Baroque composition carrying a chain of mechanisms across an oil-darkened cast-iron plate; an articulated gripper poised over a precision gear train ending in a polished bearing; a punched brass card joined to woven copper cable; cream ceramic signal insulators rhythmically crossing the rear edge; a heavy brown curtain billows into cavernous shadow while a high left glint breaks across steel, oily brass, woven wire and chalky ceramic, coarse impasto and monumental repetition; no people, no readable text.",
#     },
#     {
#         "id": "mathematics_computation",
#         "science": "mathematics; computational theory; geometry and topology",
#         "prompt": "RIJKSOIL, a medium-wide cabinet-like scholarly Baroque still life unfolding around open brass compasses on a broad chalk-dusted slate; a wooden polyhedron, faint non-readable geometric diagrams, ivory counting rods, exposed calculator wheels and a dark topology loop over folded graph parchment; moss-green baize and aged parchment form quiet vertical layers; angled candlelight, measured geometry, slate black, ivory, worn brass and wood, contemplative chiaroscuro and softly glazed surfaces; no people, no readable text.",
#     },
#     {
#         "id": "computer_science_ai_vision",
#         "science": "computer science; artificial intelligence; computer vision and pattern recognition; information systems",
#         "prompt": "RIJKSOIL, a medium-wide symmetrical Baroque nocturne built around an antique camera lens like a mechanical eye; layered cobalt circuit boards rise behind its toothed blackened-brass housing; cream punched cards meet a glass field of restrained square lights while branching gold conductors spread across the lower plane; a cool square illumination from the left balances one warm copper gleam, lacquer blue, amber glass, centralized drama, deep glazing and luminous accents; no people, no readable text.",
#     },
#     {
#         "id": "operations_economics",
#         "science": "management science and operations research; economics and econometrics; accounting",
#         "prompt": "RIJKSOIL, a medium-wide Dutch Golden Age merchant-table composition ascending from coin stacks to a brass balance beam; an oxblood leather ledger lies open on a shallow writing slope beside a dark abacus, cargo miniatures, a clear sand timer and folded sheets bearing non-readable curves; tobacco-brown drapery gathers into one generous fold; warm candlelight multiplies across tarnished silver, copper, rubbed leather, paper and dark wood in pyramidal order and sober chiaroscuro; no people, no readable text.",
#     },
#     {
#         "id": "strategy_politics_relations",
#         "science": "strategy and management; political science and international relations",
#         "prompt": "RIJKSOIL, a medium-wide courtly Baroque still life leading opposing ebony and ivory chess pieces toward a small terrestrial globe; a brass compass opens over an unreadable coastal chart on a cherrywood campaign box; treaty ribbons, red sealing wax and restrained crimson threads connect colored map pins; dark carmine damask swells behind the globe, theatrical left candlelight catches wax, silk and brass, dramatic diagonals and sumptuous glazing; no people, no readable text.",
#     },
#     {
#         "id": "sociology_philosophy",
#         "science": "sociology; political science; philosophy, knowledge and ethics",
#         "prompt": "RIJKSOIL, a medium-wide civic vanitas arranged around a shallow pewter bowl of voting tokens on a cracked black-marble ledge; clustered wooden figures of varied heights stand among census tally sticks, three linked rings and an open illegible leather book weighted by a river stone; a dark convex mirror and small brass balance catch one severe beeswax candle; smoke-gray linen and olive velvet descend into enveloping shadow, worn wood, fibrous paper, dull pewter and grave translucent glazes; no people, no readable text.",
#     },
#     {
#         "id": "psychology_cognitive_science",
#         "science": "clinical and social psychology; psychiatry and mental health; cognitive neuroscience",
#         "prompt": "RIJKSOIL, a medium-wide asymmetrical Baroque arrangement orbiting a pale ivory wax brain and a reflected theatrical mask; a wooden maze aligns with a slender metronome, ambiguous ink cards scatter among memory beads, and a silver tuning fork crosses the foreground; plum felt, pale maple and a dusky violet curtain open onto a narrow black recess; soft divided light, theatrical doubling, velvety shadows and layered oil color; no people, no readable text.",
#     },
#     {
#         "id": "public_environmental_health",
#         "science": "public, environmental and occupational health; epidemiology; general health professions",
#         "prompt": "RIJKSOIL, a medium-wide field-kit Baroque still life spreading practical instruments in a calm arc from an opened galvanized case; a brass air-sampling pump and pleated filter, a small respirator, worn leather glove, clear water vial, silver thermometer and an epidemiological map with colored pins but no labels; deep green canvas rises behind them with dust and one water stain; clear left window light reveals particles across metal, fabric and glass, earthy realism and weighty forms; no people, no readable text.",
#     },
#     {
#         "id": "neuroscience_physiology_cardiovascular",
#         "science": "neuroscience and neurology; physiology; endocrinology, diabetes and metabolism; cardiology and cardiovascular medicine",
#         "prompt": "RIJKSOIL, a medium-wide anatomical Baroque arc joining an ivory wax brain to refined wax models of a heart and paired lungs; a delicate electrode crown sends red and blue nerve threads toward a coiled brass stethoscope, a clear insulin vial, a reflex hammer and a ruby pulse watch; indigo cloth crosses a burgundy leather case under warm silver light, non-gory sculptural modeling, deep recession, luminous glass and humane layered oil glazes; no people, no readable text.",
#     },
#     {
#         "id": "oncology_immunity_pathology",
#         "science": "cancer research and oncology; hematology; immunology; pathology and forensic medicine",
#         "prompt": "RIJKSOIL, a medium-wide non-gory laboratory vanitas rising from a ruby glass dish toward an angled brass microscope; translucent red droplets, pathology slides, branching ivory antibody forms and pale cell spheres gather around a closed black specimen box; clear glass rests over dark crimson cloth against a black-burgundy recess; sharp left light turns the microscope rim gold and the slides luminous, cavernous shadow, transparent glazes and precise impasto; no people, no readable text.",
#     },
#     {
#         "id": "genetics_evolution_ecology",
#         "science": "molecular and cell biology; genetics; infectious diseases; evolution, ecology, behavior, food and plant science",
#         "prompt": "RIJKSOIL, a medium-wide naturalist's Baroque crescent sweeping from a glass double helix and abstract petri colonies toward a fossil ammonite, spiral shells, a pressed fern, seed pods and a sliced heritage pear; translucent cell vesicles mingle with a pale finch skull and dark beetle on a weathered sandstone shelf; forest-brown and green-black drapery frames cool glass and autumnal fruit, tactile bone, ribbed shell, leaf, seed and moist flesh in layered glazes; no people, no readable text.",
#     },
#     {
#         "id": "toxicology_chemistry_sustainable_materials",
#         "science": "health, toxicology and mutagenesis; chemistry and spectroscopy; biomaterials; polymers; water science; renewable energy and sustainability; materials chemistry",
#         "prompt": "RIJKSOIL, a medium-wide vertical alchemical Baroque still life rising through a coiled glass alembic with amber reagent drops and descending across a spectroscopy prism, charred leaf, clear polymer film, porous biomaterial mesh, blue solar cell, copper battery plate and pure-water vial; pale soapstone bears old amber rings beneath burnt-orange fabric and a tar-black wall; firelit left illumination refracts through glass and oxidized copper, rich layered paint and fiery chiaroscuro; no people, no readable text.",
#     },
# ]


## 3. GPU, repository, and FLUX dependencies

Only GlueStick is imported from the SAGE repository. The renderer remains
the project's pinned FLUX.2 environment.


In [ ]:
import platform
import subprocess
import sys
from pathlib import Path

print({"python": sys.version, "platform": platform.platform()})
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch is missing; use a Colab GPU runtime.") from error
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU runtime is required.")
print({"gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda})

project_path = Path(PROJECT_ROOT)
if not (project_path / "pyproject.toml").is_file():
    subprocess.check_call(["git", "clone", "--depth", "1", REPOSITORY_URL, PROJECT_ROOT])
elif UPDATE_REPOSITORY:
    subprocess.check_call(["git", "-C", PROJECT_ROOT, "pull", "--ff-only"])

core_probe = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import cv2, numpy, scipy, transformers, pydantic; "
            "from diffusers import Flux2KleinPipeline"
        ),
    ],
    capture_output=True,
    text=True,
)
if core_probe.returncode != 0:
    print(core_probe.stderr[-2000:])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-r",
        str(project_path / "requirements-colab.txt"),
    ])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", PROJECT_ROOT])
    raise RuntimeError(
        "Dependencies installed successfully. Restart the kernel once, then rerun from section 1."
    )

# GlueStick imports pytlsd at module import time. Install and verify it in
# this exact kernel interpreter before the SAGE subprocess is ever started.
sage_probe = subprocess.run(
    [sys.executable, "-c", "import omegaconf, pytlsd; from pytlsd import lsd"],
    capture_output=True,
    text=True,
)
if sage_probe.returncode != 0:
    print("Repairing missing SAGE line-detector dependencies...")
    print(sage_probe.stderr[-2000:])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "setuptools>=69", "wheel", "pybind11>=2.10",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--no-cache-dir",
        "omegaconf==2.3.0", "pytlsd==0.0.2",
    ])
    import importlib
    importlib.invalidate_caches()
    subprocess.check_call([
        sys.executable, "-c",
        "import omegaconf, pytlsd; from pytlsd import lsd; print(pytlsd.__file__)",
    ])

import importlib
package_source = str(project_path / "src")
if package_source not in sys.path:
    sys.path.insert(0, package_source)
importlib.invalidate_caches()
import flowmorph_klein
from diffusers import Flux2KleinPipeline

project_commit = subprocess.check_output(
    ["git", "-C", PROJECT_ROOT, "rev-parse", "HEAD"], text=True
).strip()
print({
    "repository_commit": project_commit,
    "flowmorph_source": flowmorph_klein.__file__,
})


## 4. Mount Drive and reserve a persistent run

No additional video-model credential is required: SAGE structures are
rendered by the already configured FLUX.2 Klein pipeline.


In [ ]:
import json
import re
from datetime import datetime, timezone

if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*", PROJECT_NAME):
    raise ValueError("PROJECT_NAME may contain only letters, numbers, underscores, and hyphens")

DRIVE_ENABLED = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_base = Path(DRIVE_PROJECT_BASE)
    drive_base.mkdir(parents=True, exist_ok=True)
    DRIVE_ENABLED = True
else:
    drive_base = None

def reserve_numbered_run(parent, project_name):
    project_root = Path(parent) / project_name
    project_root.mkdir(parents=True, exist_ok=True)
    numbers = []
    prefix = f"{project_name}_"
    for candidate in project_root.iterdir():
        if candidate.is_dir() and candidate.name.startswith(prefix):
            token = candidate.name[len(prefix):].split("_", 1)[0]
            if token.isdigit():
                numbers.append(int(token))
    sequence = max(numbers, default=0) + 1
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    while True:
        candidate = project_root / f"{project_name}_{sequence:04d}_{timestamp}"
        try:
            candidate.mkdir(parents=False, exist_ok=False)
        except FileExistsError:
            sequence += 1
            continue
        return candidate

if RESUME_RUN_DIRECTORY is not None:
    RUN_DIRECTORY = Path(RESUME_RUN_DIRECTORY).expanduser()
    if not RUN_DIRECTORY.is_dir():
        raise FileNotFoundError(f"RESUME_RUN_DIRECTORY does not exist: {RUN_DIRECTORY}")
elif DRIVE_ENABLED:
    RUN_DIRECTORY = reserve_numbered_run(drive_base, PROJECT_NAME)
else:
    RUN_DIRECTORY = reserve_numbered_run(LOCAL_ASSET_ROOT, PROJECT_NAME)

for child in ("base_frames", "trials", "previews", "metadata", "sage", "video"):
    (RUN_DIRECTORY / child).mkdir(parents=True, exist_ok=True)
Path(HF_CACHE_DIR).mkdir(parents=True, exist_ok=True)

run_identity = {
    "project": PROJECT_NAME,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "persistent": DRIVE_ENABLED,
    "run_directory": str(RUN_DIRECTORY),
    "generative_backend": MODEL_ID,
    "lora_source": LORA_SOURCE,
}
(RUN_DIRECTORY / "metadata" / "run_identity.json").write_text(
    json.dumps(run_identity, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print("Run directory:", RUN_DIRECTORY)


## 5. Validate the creative and SAGE contracts


In [ ]:
if BASE_PROMPT_COUNT is None:
    BASE_PROMPT_COUNT = len(BASE_STAGES)
elif not 3 <= BASE_PROMPT_COUNT <= len(BASE_STAGES):
    raise ValueError(f"BASE_PROMPT_COUNT must be between 3 and {len(BASE_STAGES)}")
ACTIVE_BASE_STAGES = BASE_STAGES[:BASE_PROMPT_COUNT]

if not (256 <= IMAGE_WIDTH <= 2048 and IMAGE_WIDTH % 16 == 0):
    raise ValueError("IMAGE_WIDTH must be 256–2048 and divisible by 16")
if not (256 <= IMAGE_HEIGHT <= 2048 and IMAGE_HEIGHT % 16 == 0):
    raise ValueError("IMAGE_HEIGHT must be 256–2048 and divisible by 16")
if not 1 <= IMAGE_INFERENCE_STEPS <= 100:
    raise ValueError("IMAGE_INFERENCE_STEPS must be between 1 and 100")
if not 0 <= IMAGE_GUIDANCE_SCALE <= 20:
    raise ValueError("IMAGE_GUIDANCE_SCALE must be between 0 and 20")
if not 0 < IMAGE_LORA_SCALE <= 4:
    raise ValueError("IMAGE_LORA_SCALE must lie in (0, 4]")
if not 0 <= BASE_REFERENCE_GRAIN_STRENGTH <= 0.25:
    raise ValueError("BASE_REFERENCE_GRAIN_STRENGTH must lie in [0, 0.25]")
if not 0 < BASE_REFERENCE_DENOISE_STRENGTH <= 1:
    raise ValueError("BASE_REFERENCE_DENOISE_STRENGTH must lie in (0, 1]")
if not 32 <= FLUX_PROMPT_MAX_SEQUENCE_LENGTH <= 512:
    raise ValueError("FLUX_PROMPT_MAX_SEQUENCE_LENGTH must lie in [32, 512]")

ids = [item["id"] for item in ACTIVE_BASE_STAGES]
if len(ids) != len(set(ids)) or any(not re.fullmatch(r"[a-z0-9_]+", item) for item in ids):
    raise ValueError("Anchor IDs must be unique lowercase snake_case values")
for item in ACTIVE_BASE_STAGES:
    if not item["science"].strip() or not item["prompt"].strip():
        raise ValueError(f"Blank science or prompt in {item['id']}")
    if item["prompt"].casefold().count(LORA_TRIGGER.casefold()) != 1:
        raise ValueError(f"{item['id']} must contain the LoRA trigger exactly once")

if SAGE_MASK_MODE not in {"grabcut", "full_frame", "directory"}:
    raise ValueError("SAGE_MASK_MODE must be grabcut, full_frame, or directory")
if SAGE_MASK_MODE == "directory" and not SAGE_MASK_SOURCE_DIRECTORY:
    raise ValueError("directory mask mode requires SAGE_MASK_SOURCE_DIRECTORY")
if not 0 < SAGE_MASK_MIN_COVERAGE < SAGE_MASK_MAX_COVERAGE <= 1:
    raise ValueError("Invalid SAGE mask coverage interval")
if SAGE_WIDTH % 64 or SAGE_HEIGHT % 64:
    raise ValueError("SAGE_WIDTH and SAGE_HEIGHT must be divisible by 64")
if SAGE_GENERATED_FRAMES_PER_GAP < 5:
    raise ValueError("SAGE needs at least five generated frames per gap")
if (SAGE_WIDTH, SAGE_HEIGHT) != (IMAGE_WIDTH, IMAGE_HEIGHT):
    raise ValueError("SAGE guides and FLUX output must use the same dimensions")
if not 1 <= SAGE_FLUX_INFERENCE_STEPS <= 100:
    raise ValueError("SAGE_FLUX_INFERENCE_STEPS must lie in [1, 100]")
if not 0 < SAGE_FLUX_IMG2IMG_STRENGTH <= 1:
    raise ValueError("SAGE_FLUX_IMG2IMG_STRENGTH must lie in (0, 1]")
if not 0 <= SAGE_PREVIOUS_FRAME_BLEND <= 1:
    raise ValueError("SAGE_PREVIOUS_FRAME_BLEND must lie in [0, 1]")
if not 0 <= SAGE_STRUCTURE_INIT_STRENGTH <= 1:
    raise ValueError("SAGE_STRUCTURE_INIT_STRENGTH must lie in [0, 1]")
if not 0 <= SAGE_INIT_GRAIN_STRENGTH <= 0.25:
    raise ValueError("SAGE_INIT_GRAIN_STRENGTH must lie in [0, 0.25]")

print({
    "anchors": BASE_PROMPT_COUNT,
    "cyclic_gaps": BASE_PROMPT_COUNT,
    "sage_generated_frames_per_gap": SAGE_GENERATED_FRAMES_PER_GAP,
    "exact_anchor_plus_sage_frames_per_gap": SAGE_GENERATED_FRAMES_PER_GAP + 1,
    "final_cyclic_frames": BASE_PROMPT_COUNT * (SAGE_GENERATED_FRAMES_PER_GAP + 1),
    "size": [SAGE_WIDTH, SAGE_HEIGHT],
    "renderer": MODEL_ID,
    "renderer_lora": LORA_SOURCE,
    "flux_img2img_strength": SAGE_FLUX_IMG2IMG_STRENGTH,
    "still_motion_fallback": {
        "synthetic_flow_scale": SAGE_SYNTHETIC_FLOW_SCALE,
        "trajectory_bend": SAGE_TRAJECTORY_BEND,
    },
})
print("Anchor order:", " → ".join(ids), "→", ids[0])


## 6. Load RIJKSOIL and optionally render one anchor trial


In [ ]:
import gc
import os
import random
import shutil
from huggingface_hub import hf_hub_download
from IPython.display import Markdown, display
from PIL import Image, ImageFilter
from flowmorph_klein.lora import load_flux2_lora
from flowmorph_klein.trajectory import prepare_flux2_klein_img2img_inputs

try:
    import peft.tuners.lora.torchao as peft_torchao_dispatch
except ImportError:
    peft_torchao_dispatch = None
else:
    peft_torchao_dispatch.is_torchao_available = lambda: False

downloaded_lora = Path(hf_hub_download(
    repo_id=LORA_SOURCE,
    filename=LORA_WEIGHT_NAME,
    revision=LORA_REVISION,
    cache_dir=HF_CACHE_DIR,
))
lora_stage_directory = Path(HF_CACHE_DIR) / "flowmorph_lora_files" / LORA_REVISION[:12]
lora_stage_directory.mkdir(parents=True, exist_ok=True)
LOCAL_LORA_PATH = lora_stage_directory / LORA_WEIGHT_NAME
if not LOCAL_LORA_PATH.is_file():
    try:
        os.link(downloaded_lora.resolve(), LOCAL_LORA_PATH)
    except OSError:
        shutil.copy2(downloaded_lora, LOCAL_LORA_PATH)
if LOCAL_LORA_PATH.stat().st_size != downloaded_lora.stat().st_size:
    raise RuntimeError(f"Staged LoRA size mismatch at {LOCAL_LORA_PATH}")

def release_flux_pipeline():
    previous = globals().pop("FLUX_PIPE", None)
    globals().pop("FLUX_PIPE_LORA_SCALE", None)
    if previous is not None:
        maybe_free = getattr(previous, "maybe_free_model_hooks", None)
        if callable(maybe_free):
            maybe_free()
        del previous
        gc.collect()
        torch.cuda.empty_cache()

def load_flux_pipeline():
    pipeline = Flux2KleinPipeline.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        cache_dir=HF_CACHE_DIR,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    report = load_flux2_lora(
        pipeline,
        str(LOCAL_LORA_PATH),
        adapter_name=LORA_ADAPTER_NAME,
        scale=IMAGE_LORA_SCALE,
        require_base_9b_provenance=False,
        allow_distilled_9b=True,
    )
    pipeline.fuse_lora(
        components=["transformer"],
        lora_scale=1.0,
        safe_fusing=True,
        adapter_names=[LORA_ADAPTER_NAME],
    )
    pipeline.unload_lora_weights()
    remaining = [
        name for name, _ in pipeline.transformer.named_parameters()
        if "lora_" in name.casefold() or ".lora" in name.casefold()
    ]
    if remaining:
        raise RuntimeError("LoRA fusion left runtime parameters: " + ", ".join(remaining[:5]))
    pipeline.enable_model_cpu_offload()
    pipeline.vae.enable_slicing()
    pipeline.vae.enable_tiling()
    return pipeline, report

if "FLUX_PIPE" in globals() and globals().get("FLUX_PIPE_LORA_SCALE") != float(IMAGE_LORA_SCALE):
    print("LoRA scale changed; rebuilding the fused pipeline.")
    release_flux_pipeline()
if "FLUX_PIPE" not in globals():
    FLUX_PIPE, LORA_REPORT = load_flux_pipeline()
    FLUX_PIPE_LORA_SCALE = float(IMAGE_LORA_SCALE)
    print("Loaded a device-safe fused-LoRA pipeline.")
else:
    print("Reusing the fused pipeline at the current LoRA scale.")


FLUX_PROMPT_TOKENIZER = FLUX_PIPE.tokenizer

def flux_prompt_token_count(prompt):
    messages = [{"role": "user", "content": prompt}]
    templated = FLUX_PROMPT_TOKENIZER.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    encoded = FLUX_PROMPT_TOKENIZER(
        templated,
        add_special_tokens=False,
        truncation=False,
    )
    return len(encoded["input_ids"])

def validate_flux_prompt_length(prompt, label="Prompt"):
    token_count = flux_prompt_token_count(prompt)
    if token_count > FLUX_PROMPT_MAX_SEQUENCE_LENGTH:
        raise ValueError(
            f"{label} tokenizes to {token_count} tokens after the FLUX chat "
            f"template; maximum is {FLUX_PROMPT_MAX_SEQUENCE_LENGTH}"
        )
    return token_count

if RUN_TRIAL_KEYFRAME:
    system_random = random.SystemRandom()
    trial_index = (
        TRIAL_KEYFRAME_INDEX
        if TRIAL_KEYFRAME_INDEX is not None
        else system_random.randrange(len(ACTIVE_BASE_STAGES))
    )
    if not 0 <= trial_index < len(ACTIVE_BASE_STAGES):
        raise IndexError("TRIAL_KEYFRAME_INDEX is outside the active anchor range")
    trial_seed = TRIAL_SEED if TRIAL_SEED is not None else system_random.randrange(2**31)
    trial_stage = ACTIVE_BASE_STAGES[trial_index]
    validate_flux_prompt_length(trial_stage["prompt"], "Trial anchor prompt")
    trial_result = FLUX_PIPE(
        prompt=trial_stage["prompt"],
        height=IMAGE_HEIGHT,
        width=IMAGE_WIDTH,
        num_inference_steps=IMAGE_INFERENCE_STEPS,
        guidance_scale=IMAGE_GUIDANCE_SCALE,
        generator=torch.Generator(device="cuda").manual_seed(trial_seed),
        output_type="pil",
        max_sequence_length=FLUX_PROMPT_MAX_SEQUENCE_LENGTH,
    )
    trial_image = trial_result.images[0].convert("RGB")
    trial_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    trial_directory = RUN_DIRECTORY / "trials" / f"{trial_stamp}_{trial_stage['id']}_{trial_seed}"
    trial_directory.mkdir(parents=True, exist_ok=False)
    trial_path = trial_directory / "trial.png"
    trial_image.save(trial_path)
    (trial_directory / "settings.json").write_text(json.dumps({
        "stage": trial_stage,
        "seed": trial_seed,
        "lora_scale": IMAGE_LORA_SCALE,
        "guidance_scale": IMAGE_GUIDANCE_SCALE,
        "inference_steps": IMAGE_INFERENCE_STEPS,
        "size": [IMAGE_WIDTH, IMAGE_HEIGHT],
    }, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    preview = trial_image.copy()
    preview.thumbnail((TRIAL_DISPLAY_MAX_WIDTH, TRIAL_DISPLAY_MAX_WIDTH))
    display(Markdown(f"### Trial anchor: `{trial_stage['id']}`"))
    display(preview)
    print({"path": str(trial_path), "seed": trial_seed, "prompt_index": trial_index})
    del trial_result, trial_image, preview
else:
    print("Trial skipped.")


## 7. Generate the softly related cyclic anchor paintings

As in the prompt-only workflow, each later anchor can use a blurred,
grained previous painting as weak latent img2img initialization. There is
no mask canvas and no fixed beige background construction.


In [ ]:
from flowmorph_klein.art_loop import make_soft_reference

BASE_DIRECTORY = RUN_DIRECTORY / "base_frames"
REFERENCE_DIRECTORY = BASE_DIRECTORY / "soft_references"
BASE_MANIFEST_PATH = RUN_DIRECTORY / "metadata" / "base_manifest.json"
BASE_RECORDS = []

def generate_prompt_anchor(prompt, seed, reference=None):
    validate_flux_prompt_length(prompt, "Anchor generation prompt")
    generator = torch.Generator(device="cuda").manual_seed(seed)
    kwargs = {
        "prompt": prompt,
        "height": IMAGE_HEIGHT,
        "width": IMAGE_WIDTH,
        "num_inference_steps": IMAGE_INFERENCE_STEPS,
        "guidance_scale": IMAGE_GUIDANCE_SCALE,
        "generator": generator,
        "output_type": "pil",
        "max_sequence_length": FLUX_PROMPT_MAX_SEQUENCE_LENGTH,
    }
    generation_report = {
        "mode": "text_to_image",
        "requested_img2img_strength": None,
        "effective_start_sigma": None,
    }
    if reference is not None:
        generation_inputs = prepare_flux2_klein_img2img_inputs(
            FLUX_PIPE,
            reference,
            width=IMAGE_WIDTH,
            height=IMAGE_HEIGHT,
            num_inference_steps=IMAGE_INFERENCE_STEPS,
            strength=BASE_REFERENCE_DENOISE_STRENGTH,
            generator=generator,
        )
        kwargs["sigmas"] = list(generation_inputs.sigmas)
        kwargs["latents"] = generation_inputs.latents
        generation_report = {
            "mode": "latent_img2img_from_weak_previous_reference",
            "requested_img2img_strength": (
                generation_inputs.requested_strength
            ),
            "effective_start_sigma": generation_inputs.effective_start_sigma,
            "denoising_steps": generation_inputs.denoising_steps,
        }
    result = FLUX_PIPE(**kwargs)
    if not result.images:
        raise RuntimeError("FLUX returned no anchor image")
    return result.images[0].convert("RGB"), generation_report

if not REGENERATE_BASE_FRAMES and BASE_MANIFEST_PATH.is_file():
    BASE_RECORDS = json.loads(
        BASE_MANIFEST_PATH.read_text(encoding="utf-8")
    )["records"]
    missing = [
        item["path"]
        for item in BASE_RECORDS
        if not Path(item["path"]).is_file()
    ]
    if missing:
        raise FileNotFoundError(
            "Missing resumed anchor images: " + ", ".join(missing)
        )
    resumed_contract = [
        (record["uid"], record["science"], record["prompt"])
        for record in BASE_RECORDS
    ]
    current_contract = [
        (f"base_{index:03d}", stage["science"], stage["prompt"])
        for index, stage in enumerate(ACTIVE_BASE_STAGES)
    ]
    if resumed_contract != current_contract:
        raise RuntimeError(
            "Editable anchor prompts differ from the saved anchors. "
            "Regenerate or resume the matching run."
        )
    print(f"Loaded {len(BASE_RECORDS)} existing anchor records.")
else:
    BASE_DIRECTORY.mkdir(parents=True, exist_ok=True)
    previous = None
    for index, stage in enumerate(ACTIVE_BASE_STAGES):
        seed = BASE_SEED + index
        reference = None
        reference_path = None
        if previous is not None and BASE_CONTINUITY_ENABLED:
            reference = make_soft_reference(
                previous,
                # A blend of 1.0 means 100% blurred previous image. No
                # fixed beige/gray background canvas contributes.
                reference_blend=1.0,
                blur_radius=BASE_REFERENCE_BLUR,
                grain_strength=BASE_REFERENCE_GRAIN_STRENGTH,
                grain_seed=seed,
            )
            if SAVE_SOFT_REFERENCES:
                REFERENCE_DIRECTORY.mkdir(parents=True, exist_ok=True)
                reference_path = (
                    REFERENCE_DIRECTORY / f"reference_{index:03d}.png"
                )
                reference.save(reference_path, format="PNG", compress_level=4)
        image, generation_report = generate_prompt_anchor(
            stage["prompt"],
            seed,
            reference=reference,
        )
        output_path = BASE_DIRECTORY / f"{index:03d}_{stage['id']}.png"
        image.save(output_path, format="PNG", compress_level=4)
        record = {
            "uid": f"base_{index:03d}",
            "kind": "base",
            "round": 0,
            "science": stage["science"],
            "prompt": stage["prompt"],
            "generation_prompt": stage["prompt"],
            "generation_prompt_token_count": validate_flux_prompt_length(
                stage["prompt"],
                "Saved anchor prompt",
            ),
            "seed": seed,
            "path": str(output_path),
            "soft_reference_path": (
                str(reference_path) if reference_path else None
            ),
            "base_continuity_used": reference is not None,
            "base_reference_source": (
                "blurred_grained_previous_without_flat_canvas"
            ),
            "base_reference_blur": BASE_REFERENCE_BLUR,
            "base_reference_grain_strength": (
                BASE_REFERENCE_GRAIN_STRENGTH
            ),
            "generation_mode": generation_report["mode"],
            "img2img_strength": generation_report[
                "requested_img2img_strength"
            ],
            "effective_start_sigma": generation_report[
                "effective_start_sigma"
            ],
        }
        BASE_RECORDS.append(record)
        BASE_MANIFEST_PATH.write_text(json.dumps({
            "project": PROJECT_NAME,
            "complete": len(BASE_RECORDS) == len(ACTIVE_BASE_STAGES),
            "records": BASE_RECORDS,
        }, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
        if previous is not None:
            previous.close()
        previous = image.copy()
        image.close()
        if reference is not None:
            reference.close()
        print(
            f"Anchor {index + 1}/{len(ACTIVE_BASE_STAGES)} saved: "
            f"{output_path.name}"
        )
    if previous is not None:
        previous.close()

if len(BASE_RECORDS) != len(ACTIVE_BASE_STAGES):
    raise RuntimeError("The anchor manifest is incomplete.")
print(f"Prepared {len(BASE_RECORDS)} cyclic anchors in {BASE_DIRECTORY}")


In [ ]:
from flowmorph_klein.visualization import make_contact_sheet

base_contact_sheet_path = RUN_DIRECTORY / "previews" / "base_contact_sheet.png"
base_images = [Image.open(item["path"]).convert("RGB") for item in BASE_RECORDS]
make_contact_sheet(
    base_images,
    base_contact_sheet_path,
    columns=min(CONTACT_SHEET_COLUMNS, len(base_images)),
    labels=[item["uid"] for item in BASE_RECORDS],
)
for image in base_images:
    image.close()
base_preview = Image.open(base_contact_sheet_path).convert("RGB")
base_preview.thumbnail((CONTACT_SHEET_DISPLAY_MAX_WIDTH, 100000))
display(Markdown("### Anchor paintings — compact contact sheet"))
display(base_preview)
del base_preview, base_images
print("Full-resolution anchors and contact sheet:", BASE_DIRECTORY)

saved_reference_paths = [
    Path(item["soft_reference_path"])
    for item in BASE_RECORDS
    if item.get("soft_reference_path") and Path(item["soft_reference_path"]).is_file()
]
if saved_reference_paths:
    reference_contact_sheet_path = (
        RUN_DIRECTORY / "previews" / "anchor_soft_reference_contact_sheet.png"
    )
    reference_images = []
    for path in saved_reference_paths:
        with Image.open(path) as opened:
            thumbnail = opened.convert("RGB")
            thumbnail.thumbnail((192, 192))
            reference_images.append(thumbnail)
    make_contact_sheet(
        reference_images,
        reference_contact_sheet_path,
        columns=min(CONTACT_SHEET_COLUMNS, len(reference_images)),
        labels=[path.stem for path in saved_reference_paths],
    )
    for image in reference_images:
        image.close()
    reference_preview = Image.open(reference_contact_sheet_path).convert("RGB")
    reference_preview.thumbnail((CONTACT_SHEET_DISPLAY_MAX_WIDTH, 100000))
    display(Markdown("### Blurred/grained anchor initialization images"))
    display(reference_preview)
    reference_preview.close()
    print("Full-resolution anchor initialization images:", saved_reference_paths[0].parent)


## 8. Build or load foreground masks and inspect them

SAGE deliberately suppresses background line clutter by matching only
lines intersecting the foreground mask. White means foreground. Automatic
GrabCut is a convenience, not ground truth: replace any bad mask PNG and
rerun from section 9 with `SAGE_MASK_REGENERATE=False`.


In [ ]:
import cv2
import numpy as np
from PIL import ImageOps

SAGE_MASK_DIRECTORY = RUN_DIRECTORY / "sage" / "masks"
SAGE_MASK_DIRECTORY.mkdir(parents=True, exist_ok=True)
SAGE_MASK_RECORDS = []

def automatic_grabcut_mask(image):
    rgb = np.asarray(image.convert("RGB"), dtype=np.uint8)
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    height, width = bgr.shape[:2]
    margin_x = max(1, round(width * SAGE_GRABCUT_MARGIN_FRACTION))
    margin_y = max(1, round(height * SAGE_GRABCUT_MARGIN_FRACTION))
    rectangle = (
        margin_x,
        margin_y,
        max(2, width - 2 * margin_x),
        max(2, height - 2 * margin_y),
    )
    labels = np.zeros((height, width), dtype=np.uint8)
    background_model = np.zeros((1, 65), dtype=np.float64)
    foreground_model = np.zeros((1, 65), dtype=np.float64)
    cv2.grabCut(
        bgr,
        labels,
        rectangle,
        background_model,
        foreground_model,
        5,
        cv2.GC_INIT_WITH_RECT,
    )
    foreground = np.isin(labels, [cv2.GC_FGD, cv2.GC_PR_FGD]).astype(np.uint8)
    if SAGE_MASK_DILATION_PIXELS > 0:
        size = int(SAGE_MASK_DILATION_PIXELS) * 2 + 1
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))
        foreground = cv2.morphologyEx(foreground, cv2.MORPH_CLOSE, kernel)
        foreground = cv2.dilate(foreground, kernel, iterations=1)
    coverage = float(foreground.mean())
    if not SAGE_MASK_MIN_COVERAGE <= coverage <= SAGE_MASK_MAX_COVERAGE:
        print(
            f"GrabCut coverage {coverage:.3f} is outside the requested range; "
            "using full frame for this anchor."
        )
        foreground = np.ones((height, width), dtype=np.uint8)
    return Image.fromarray(foreground * 255, mode="L")

for record in BASE_RECORDS:
    output_path = SAGE_MASK_DIRECTORY / f"{record['uid']}.png"
    if output_path.is_file() and not SAGE_MASK_REGENERATE:
        mask = Image.open(output_path).convert("L")
        source = "existing_run_mask"
    elif SAGE_MASK_MODE == "directory":
        input_path = Path(SAGE_MASK_SOURCE_DIRECTORY).expanduser() / f"{record['uid']}.png"
        if not input_path.is_file():
            raise FileNotFoundError(f"Missing SAGE mask: {input_path}")
        with Image.open(input_path) as opened:
            mask = opened.convert("L")
        source = str(input_path)
    elif SAGE_MASK_MODE == "full_frame":
        with Image.open(record["path"]) as opened:
            mask = Image.new("L", opened.size, 255)
        source = "full_frame"
    else:
        with Image.open(record["path"]) as opened:
            mask = automatic_grabcut_mask(opened)
        source = "automatic_grabcut"
    mask.save(output_path)
    coverage = float((np.asarray(mask, dtype=np.uint8) >= 128).mean())
    mask.close()
    SAGE_MASK_RECORDS.append({
        "uid": record["uid"],
        "mask_path": str(output_path),
        "source": source,
        "coverage": coverage,
    })

mask_by_uid = {item["uid"]: item for item in SAGE_MASK_RECORDS}
SAGE_ANCHOR_MANIFEST_PATH = RUN_DIRECTORY / "metadata" / "sage_anchor_manifest.json"
SAGE_ANCHOR_MANIFEST = {
    "cyclic": True,
    "mask_mode": SAGE_MASK_MODE,
    "anchors": [
        {
            **record,
            "mask_path": mask_by_uid[record["uid"]]["mask_path"],
            "mask_coverage": mask_by_uid[record["uid"]]["coverage"],
        }
        for record in BASE_RECORDS
    ],
}
SAGE_ANCHOR_MANIFEST_PATH.write_text(
    json.dumps(SAGE_ANCHOR_MANIFEST, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

mask_contact_sheet_path = RUN_DIRECTORY / "previews" / "sage_foreground_masks.png"
mask_previews = []
for item in SAGE_MASK_RECORDS:
    with Image.open(item["mask_path"]) as opened:
        mask_previews.append(opened.convert("RGB"))
make_contact_sheet(
    mask_previews,
    mask_contact_sheet_path,
    columns=min(CONTACT_SHEET_COLUMNS, len(mask_previews)),
    labels=[f"{item['uid']} ({item['coverage']:.0%})" for item in SAGE_MASK_RECORDS],
)
for image in mask_previews:
    image.close()
mask_preview = Image.open(mask_contact_sheet_path).convert("RGB")
mask_preview.thumbnail((CONTACT_SHEET_DISPLAY_MAX_WIDTH, 100000))
display(Markdown("### SAGE foreground masks — white structures will guide matching"))
display(mask_preview)
mask_preview.close()
print("Editable full-resolution masks:", SAGE_MASK_DIRECTORY)


## 9. Install SAGE's structural frontend while retaining FLUX

SAGE contributes GlueStick matching and spline-propagated line guides.
The paper's Stable Video Diffusion/FCVG renderer is deliberately not
installed or downloaded. `FLUX_PIPE` remains the sole image renderer,
with the already fused RIJKSOIL LoRA.


In [ ]:
import gc
import shutil
import urllib.request

sage_import_probe = subprocess.run(
    [sys.executable, "-c", "import omegaconf, pytlsd; from pytlsd import lsd"],
    capture_output=True,
    text=True,
)
if sage_import_probe.returncode != 0:
    print("Repairing missing SAGE line-detector dependencies...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "setuptools>=69", "wheel", "pybind11>=2.10",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--no-cache-dir",
        "omegaconf==2.3.0", "pytlsd==0.0.2",
    ])
    import importlib
    importlib.invalidate_caches()
    subprocess.check_call([
        sys.executable, "-c",
        "import omegaconf, pytlsd; from pytlsd import lsd; print(pytlsd.__file__)",
    ])

maybe_free = getattr(FLUX_PIPE, "maybe_free_model_hooks", None)
if callable(maybe_free):
    maybe_free()
gc.collect()
torch.cuda.empty_cache()

sage_repository = Path(SAGE_REPOSITORY_DIRECTORY)
if not (sage_repository / ".git").is_dir():
    subprocess.check_call(["git", "clone", SAGE_REPOSITORY_URL, str(sage_repository)])
subprocess.check_call([
    "git", "-C", str(sage_repository), "fetch", "origin", SAGE_REPOSITORY_COMMIT,
])
subprocess.check_call([
    "git", "-C", str(sage_repository), "checkout", "--detach", SAGE_REPOSITORY_COMMIT,
])
actual_sage_commit = subprocess.check_output(
    ["git", "-C", str(sage_repository), "rev-parse", "HEAD"], text=True
).strip()
if actual_sage_commit != SAGE_REPOSITORY_COMMIT:
    raise RuntimeError("SAGE checkout does not match the pinned paper implementation")

checkpoint_root = Path(HF_CACHE_DIR) / "sage_gluestick"
checkpoint_root.mkdir(parents=True, exist_ok=True)
SAGE_GLUESTICK_CHECKPOINT = checkpoint_root / "checkpoint_GlueStick_MD.tar"
if not SAGE_GLUESTICK_CHECKPOINT.is_file():
    temporary_path = SAGE_GLUESTICK_CHECKPOINT.with_suffix(".download")
    urllib.request.urlretrieve(SAGE_GLUESTICK_URL, temporary_path)
    temporary_path.replace(SAGE_GLUESTICK_CHECKPOINT)

print({
    "sage_commit": actual_sage_commit,
    "sage_component": "GlueStick + normalized matching + spline line propagation",
    "renderer": MODEL_ID,
    "renderer_lora": LORA_SOURCE,
    "lora_scale": IMAGE_LORA_SCALE,
    "stable_video_diffusion_downloaded": False,
    "cuda_reserved_gib": round(torch.cuda.memory_reserved() / 1024**3, 3),
})


## 10. Prepare and inspect SAGE structural guides

This inexpensive subprocess loads GlueStick once, matches each circular
pair, and writes the true interior guide sequence. It does not load a
second generative model. Inspect the overlays and middle conditions
before starting the FLUX render.


In [ ]:
SAGE_OUTPUT_ROOT = RUN_DIRECTORY / "sage" / "one_round"
SAGE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SAGE_RUNNER = Path(PROJECT_ROOT) / "scripts" / "sage_still_sequence_runner.py"
if not SAGE_RUNNER.is_file():
    raise FileNotFoundError(SAGE_RUNNER)

SAGE_COMMAND = [
    sys.executable,
    str(SAGE_RUNNER),
    "--sage-repo", str(sage_repository),
    "--manifest", str(SAGE_ANCHOR_MANIFEST_PATH),
    "--output-root", str(SAGE_OUTPUT_ROOT),
    "--gluestick-checkpoint", str(SAGE_GLUESTICK_CHECKPOINT),
    "--width", str(SAGE_WIDTH),
    "--height", str(SAGE_HEIGHT),
    "--generated-frames", str(SAGE_GENERATED_FRAMES_PER_GAP),
    "--max-points", str(SAGE_MAX_POINTS),
    "--max-lines", str(SAGE_MAX_LINES),
    "--max-matched-lines", str(SAGE_MAX_MATCHED_LINES),
    "--minimum-matched-lines", str(SAGE_MINIMUM_MATCHED_LINES),
    "--line-width", str(SAGE_CONDITION_LINE_WIDTH),
    "--trajectory-bend", str(SAGE_TRAJECTORY_BEND),
    "--synthetic-flow-scale", str(SAGE_SYNTHETIC_FLOW_SCALE),
]
if SAGE_REUSE_COMPLETED_GAPS:
    SAGE_COMMAND.append("--reuse")

if callable(maybe_free):
    maybe_free()
gc.collect()
torch.cuda.empty_cache()
subprocess.check_call(SAGE_COMMAND)

SAGE_PREPARATION_MANIFEST_PATH = SAGE_OUTPUT_ROOT / "sage_preparation_manifest.json"
SAGE_PREPARATION = json.loads(
    SAGE_PREPARATION_MANIFEST_PATH.read_text(encoding="utf-8")
)

def display_path_contact_sheet(paths, labels, title, filename):
    images = [Image.open(path).convert("RGB") for path in paths]
    output_path = RUN_DIRECTORY / "previews" / filename
    make_contact_sheet(
        images,
        output_path,
        columns=min(CONTACT_SHEET_COLUMNS, len(images)),
        labels=labels,
    )
    for image in images:
        image.close()
    preview = Image.open(output_path).convert("RGB")
    preview.thumbnail((CONTACT_SHEET_DISPLAY_MAX_WIDTH, 100000))
    display(Markdown(title))
    display(preview)
    preview.close()
    return output_path

line_preview_paths = []
condition_preview_paths = []
for gap in SAGE_PREPARATION["gaps"]:
    gap_directory = Path(gap["gap_directory"])
    line_preview_paths.extend([
        gap_directory / "source_matched_lines.png",
        gap_directory / "target_matched_lines.png",
    ])
    condition_paths = [Path(path) for path in gap["condition_paths"]]
    condition_preview_paths.append(condition_paths[len(condition_paths) // 2])

display_path_contact_sheet(
    line_preview_paths,
    [path.parent.name + "/" + path.stem for path in line_preview_paths],
    "### Matched foreground lines at both sides of every gap",
    "sage_matched_line_overlays.png",
)
display_path_contact_sheet(
    condition_preview_paths,
    [path.parent.parent.name for path in condition_preview_paths],
    "### Middle SAGE structural guide in every gap",
    "sage_middle_conditions.png",
)
print("Prepared SAGE guides:", SAGE_OUTPUT_ROOT)


## 11. Render the SAGE-guided round with FLUX.2 Klein + RIJKSOIL

For every interior time step, the notebook builds a soft initialization
from low-frequency endpoint color, the previous generated frame, seeded
grain, and the current SAGE line guide. That initialization supplies true
FLUX img2img latents. Both exact endpoints and the initialization are also
passed as FLUX.2 reference images, while endpoint prompt embeddings are
interpolated at the same time coordinate. Frames and metadata are saved
immediately, and completed fingerprint-matching gaps are reusable.


In [ ]:
import hashlib
from PIL import ImageEnhance, ImageOps
from flowmorph_klein.conditioning import (
    encode_prompt_conditioning,
    interpolate_conditioning,
)
from flowmorph_klein.trajectory import prepare_flux2_klein_img2img_inputs

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def write_json_atomic(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)

def fit_frame(value):
    if isinstance(value, Image.Image):
        opened = value.convert("RGB")
    else:
        with Image.open(value) as source:
            opened = source.convert("RGB")
    fitted = ImageOps.fit(
        opened,
        (SAGE_WIDTH, SAGE_HEIGHT),
        method=Image.Resampling.LANCZOS,
    )
    if opened is not value:
        opened.close()
    return fitted

def make_sage_flux_init(left, right, condition, previous, alpha, seed):
    left_low = left.filter(ImageFilter.GaussianBlur(SAGE_ENDPOINT_PALETTE_BLUR))
    right_low = right.filter(ImageFilter.GaussianBlur(SAGE_ENDPOINT_PALETTE_BLUR))
    canvas = Image.blend(left_low, right_low, float(alpha))
    left_low.close()
    right_low.close()

    if previous is not None and SAGE_PREVIOUS_FRAME_BLEND > 0:
        previous_low = previous.filter(
            ImageFilter.GaussianBlur(SAGE_PREVIOUS_FRAME_BLUR)
        )
        canvas = Image.blend(canvas, previous_low, SAGE_PREVIOUS_FRAME_BLEND)
        previous_low.close()

    guide = condition.convert("L")
    if SAGE_STRUCTURE_DILATION_PIXELS > 0:
        kernel = SAGE_STRUCTURE_DILATION_PIXELS * 2 + 1
        guide = guide.filter(ImageFilter.MaxFilter(kernel))
    guide = guide.filter(ImageFilter.GaussianBlur(0.8))
    guide = guide.point(
        lambda value: int(round(value * SAGE_STRUCTURE_INIT_STRENGTH))
    )
    line_layer = ImageEnhance.Brightness(canvas).enhance(0.58)
    structured = Image.composite(line_layer, canvas, guide)
    line_layer.close()
    guide.close()
    canvas.close()

    if SAGE_INIT_GRAIN_STRENGTH > 0:
        array = np.asarray(structured, dtype=np.float32)
        rng = np.random.default_rng(seed)
        noise = rng.normal(
            0.0,
            255.0 * SAGE_INIT_GRAIN_STRENGTH,
            size=(SAGE_HEIGHT, SAGE_WIDTH, 1),
        )
        array = np.clip(array + noise, 0, 255).astype(np.uint8)
        structured.close()
        structured = Image.fromarray(array, mode="RGB")
    return structured

def ffmpeg_frames(input_directory, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-framerate", str(SAGE_OUTPUT_FPS),
        "-i", str(Path(input_directory) / "frame_%04d.png"),
        "-c:v", "libx264", "-preset", "slow", "-crf", str(SAGE_VIDEO_CRF),
        "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(output_path),
    ])

RENDER_CONTRACT = {
    "method": "SAGE structure + FLUX.2 Klein img2img/reference rendering",
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "lora_source": LORA_SOURCE,
    "lora_revision": LORA_REVISION,
    "lora_weight_name": LORA_WEIGHT_NAME,
    "lora_scale": IMAGE_LORA_SCALE,
    "inference_steps": SAGE_FLUX_INFERENCE_STEPS,
    "guidance_scale": SAGE_FLUX_GUIDANCE_SCALE,
    "img2img_strength": SAGE_FLUX_IMG2IMG_STRENGTH,
    "endpoint_palette_blur": SAGE_ENDPOINT_PALETTE_BLUR,
    "previous_frame_blend": SAGE_PREVIOUS_FRAME_BLEND,
    "previous_frame_blur": SAGE_PREVIOUS_FRAME_BLUR,
    "structure_init_strength": SAGE_STRUCTURE_INIT_STRENGTH,
    "structure_dilation_pixels": SAGE_STRUCTURE_DILATION_PIXELS,
    "grain_strength": SAGE_INIT_GRAIN_STRENGTH,
    "prompt_interpolation": "linear endpoint embedding interpolation",
    "reference_images": "left endpoint + right endpoint + per-frame SAGE init",
}
RENDER_CONTRACT_HASH = hashlib.sha256(
    json.dumps(RENDER_CONTRACT, sort_keys=True).encode("utf-8")
).hexdigest()

if RUN_SAGE_RENDER:
    print("Encoding each unique endpoint prompt once...")
    ANCHOR_CONDITIONING = {}
    for record in BASE_RECORDS:
        ANCHOR_CONDITIONING[record["uid"]] = encode_prompt_conditioning(
            FLUX_PIPE,
            record["prompt"],
            device=FLUX_PIPE._execution_device,
            max_sequence_length=FLUX_PROMPT_MAX_SEQUENCE_LENGTH,
        ).cpu()
    if callable(maybe_free):
        maybe_free()
    gc.collect()
    torch.cuda.empty_cache()

    COMPLETED_SAGE_GAPS = []
    for gap in SAGE_PREPARATION["gaps"]:
        gap_directory = Path(gap["gap_directory"])
        rendered_directory = gap_directory / "flux_rendered"
        init_directory = gap_directory / "flux_inits"
        complete_directory = gap_directory / "flux_complete"
        for directory in (rendered_directory, init_directory, complete_directory):
            directory.mkdir(parents=True, exist_ok=True)
        metadata_path = gap_directory / "flux_render_metadata.json"

        left_record = gap["left"]
        right_record = gap["right"]
        fingerprint = hashlib.sha256(json.dumps({
            "render_contract_hash": RENDER_CONTRACT_HASH,
            "sage_preparation_contract_hash": SAGE_PREPARATION["contract_hash"],
            "gap_uid": gap["gap_uid"],
            "endpoint_hashes": gap["endpoint_hashes"],
            "left_prompt": left_record["prompt"],
            "right_prompt": right_record["prompt"],
            "condition_hashes": [sha256_file(path) for path in gap["condition_paths"]],
        }, sort_keys=True).encode("utf-8")).hexdigest()

        saved = {}
        if metadata_path.is_file():
            saved = json.loads(metadata_path.read_text(encoding="utf-8"))
        saved_paths = [Path(path) for path in saved.get("rendered_frame_paths", [])]
        if not (
            SAGE_REUSE_COMPLETED_GAPS
            and saved.get("fingerprint") == fingerprint
            and saved.get("complete")
            and len(saved_paths) == SAGE_GENERATED_FRAMES_PER_GAP
            and all(path.is_file() for path in saved_paths)
        ):
            saved_paths = []

        left_image = fit_frame(left_record["path"])
        right_image = fit_frame(right_record["path"])
        previous_image = left_image.copy()
        rendered_paths = []
        alphas = gap["condition_alphas"]
        if len(alphas) != SAGE_GENERATED_FRAMES_PER_GAP:
            raise RuntimeError("SAGE guide count does not match configured interior frames")

        for frame_index, (alpha, condition_path) in enumerate(
            zip(alphas, gap["condition_paths"])
        ):
            output_path = rendered_directory / f"flux_sage_{frame_index:04d}.png"
            init_path = init_directory / f"init_{frame_index:04d}.png"
            if frame_index < len(saved_paths) and output_path.is_file():
                previous_image.close()
                previous_image = fit_frame(output_path)
                rendered_paths.append(str(output_path))
                continue

            with Image.open(condition_path) as opened_condition:
                condition_image = fit_frame(opened_condition)
            frame_seed = BASE_SEED + 500_000 + int(gap["gap_index"])
            init_image = make_sage_flux_init(
                left_image,
                right_image,
                condition_image,
                previous_image,
                float(alpha),
                frame_seed + frame_index,
            )
            init_image.save(init_path, format="PNG", compress_level=4)
            generator = torch.Generator(device="cuda").manual_seed(frame_seed)
            img2img = prepare_flux2_klein_img2img_inputs(
                FLUX_PIPE,
                init_image,
                width=SAGE_WIDTH,
                height=SAGE_HEIGHT,
                num_inference_steps=SAGE_FLUX_INFERENCE_STEPS,
                strength=SAGE_FLUX_IMG2IMG_STRENGTH,
                generator=generator,
            )
            prompt_package = interpolate_conditioning(
                ANCHOR_CONDITIONING[left_record["uid"]],
                ANCHOR_CONDITIONING[right_record["uid"]],
                float(alpha),
            ).to(FLUX_PIPE._execution_device, dtype=torch.bfloat16)
            result = FLUX_PIPE(
                image=[left_image, right_image, init_image],
                prompt_embeds=prompt_package.prompt_embeds,
                height=SAGE_HEIGHT,
                width=SAGE_WIDTH,
                num_inference_steps=SAGE_FLUX_INFERENCE_STEPS,
                guidance_scale=SAGE_FLUX_GUIDANCE_SCALE,
                generator=generator,
                latents=img2img.latents,
                sigmas=list(img2img.sigmas),
                output_type="pil",
                max_sequence_length=FLUX_PROMPT_MAX_SEQUENCE_LENGTH,
            )
            generated = result.images[0].convert("RGB")
            generated.save(output_path, format="PNG", compress_level=4)
            rendered_paths.append(str(output_path))
            previous_image.close()
            previous_image = generated.copy()
            generated.close()
            init_image.close()
            condition_image.close()
            del result, img2img, prompt_package

            partial = {
                "gap_uid": gap["gap_uid"],
                "gap_index": gap["gap_index"],
                "left": left_record,
                "right": right_record,
                "fingerprint": fingerprint,
                "render_contract": RENDER_CONTRACT,
                "rendered_frame_paths": rendered_paths,
                "complete": False,
            }
            write_json_atomic(metadata_path, partial)
            print(
                f"Saved FLUX SAGE frame {frame_index + 1}/"
                f"{SAGE_GENERATED_FRAMES_PER_GAP} for {gap['gap_uid']}"
            )

        complete_paths = []
        complete_images = [left_image]
        complete_images.extend(fit_frame(path) for path in rendered_paths)
        complete_images.append(right_image)
        for index, frame in enumerate(complete_images):
            destination = complete_directory / f"frame_{index:04d}.png"
            frame.save(destination, format="PNG", compress_level=4)
            complete_paths.append(str(destination))
        clip_path = gap_directory / "flux_sage_transition.mp4"
        ffmpeg_frames(complete_directory, clip_path)
        completed = {
            "gap_uid": gap["gap_uid"],
            "gap_index": gap["gap_index"],
            "left": left_record,
            "right": right_record,
            "fingerprint": fingerprint,
            "render_contract": RENDER_CONTRACT,
            "condition_alphas": alphas,
            "condition_paths": gap["condition_paths"],
            "rendered_frame_paths": rendered_paths,
            "complete_frame_paths": complete_paths,
            "clip_path": str(clip_path),
            "complete": True,
        }
        write_json_atomic(metadata_path, completed)
        COMPLETED_SAGE_GAPS.append(completed)
        for frame in complete_images:
            frame.close()
        previous_image.close()
        print("Completed FLUX-rendered gap:", clip_path)

    sequence_directory = SAGE_OUTPUT_ROOT / "flux_cyclic_frames"
    if sequence_directory.exists():
        shutil.rmtree(sequence_directory)
    sequence_directory.mkdir(parents=True, exist_ok=False)
    sequence_paths = []
    sequence_index = 0
    for gap in COMPLETED_SAGE_GAPS:
        for path in [Path(value) for value in gap["complete_frame_paths"]][:-1]:
            destination = sequence_directory / f"frame_{sequence_index:04d}.png"
            shutil.copy2(path, destination)
            sequence_paths.append(str(destination))
            sequence_index += 1
    SAGE_FINAL_VIDEO_PATH = RUN_DIRECTORY / "video" / "sage_flux2_klein_cyclic.mp4"
    ffmpeg_frames(sequence_directory, SAGE_FINAL_VIDEO_PATH)
    SAGE_SEQUENCE = {
        "method": "SAGE structure rendered by FLUX.2 Klein + RIJKSOIL",
        "paper": "https://arxiv.org/abs/2510.24667v2",
        "cyclic": True,
        "renderer": MODEL_ID,
        "lora_source": LORA_SOURCE,
        "render_contract": RENDER_CONTRACT,
        "gaps": COMPLETED_SAGE_GAPS,
        "sequence_frame_paths": sequence_paths,
        "frame_count": len(sequence_paths),
        "fps": SAGE_OUTPUT_FPS,
        "final_video_path": str(SAGE_FINAL_VIDEO_PATH),
    }
    SAGE_SEQUENCE_MANIFEST_PATH = SAGE_OUTPUT_ROOT / "sage_sequence_manifest.json"
    write_json_atomic(SAGE_SEQUENCE_MANIFEST_PATH, SAGE_SEQUENCE)
else:
    print("FLUX SAGE render intentionally stopped after guide inspection.")


## 12. Preview the FLUX-rendered SAGE result


In [ ]:
from IPython.display import Video

if RUN_SAGE_RENDER:
    midpoint_paths = []
    for gap in SAGE_SEQUENCE["gaps"]:
        rendered = [Path(path) for path in gap["rendered_frame_paths"]]
        midpoint_paths.append(rendered[len(rendered) // 2])
        if SAGE_DISPLAY_EACH_GAP_VIDEO:
            display(Markdown(
                f"### `{gap['left']['uid']}` → `{gap['right']['uid']}`"
            ))
            display(Video(
                str(gap["clip_path"]),
                embed=False,
                width=SAGE_VIDEO_DISPLAY_WIDTH,
            ))

    display_path_contact_sheet(
        midpoint_paths,
        [gap["gap_uid"] for gap in SAGE_SEQUENCE["gaps"]],
        "### FLUX.2 Klein + RIJKSOIL midpoint from every SAGE gap",
        "sage_flux_generated_midpoints.png",
    )
    display(Markdown("## Final one-round cyclic SAGE/FLUX video"))
    display(Video(
        str(SAGE_FINAL_VIDEO_PATH),
        embed=False,
        width=SAGE_VIDEO_DISPLAY_WIDTH,
        html_attributes="controls loop muted",
    ))
    print({
        "final_video": str(SAGE_FINAL_VIDEO_PATH),
        "frames": SAGE_SEQUENCE["frame_count"],
        "fps": SAGE_SEQUENCE["fps"],
        "renderer": SAGE_SEQUENCE["renderer"],
        "lora": SAGE_SEQUENCE["lora_source"],
        "cyclic": SAGE_SEQUENCE["cyclic"],
    })
    if DOWNLOAD_FINAL_VIDEO:
        from google.colab import files
        files.download(str(SAGE_FINAL_VIDEO_PATH))


## Interpretation

This is a SAGE **structure adaptation**, not a claim that FLUX.2 Klein
natively implements the paper's FCVG renderer. SAGE determines the
matched line topology and its spline trajectory. FLUX.2 Klein 9B with
RIJKSOIL renders every visible interior image from those guides. Because
the current FLUX ControlNet implementation targets FLUX.1 rather than
FLUX.2 Klein, the guide enters through a saved, inspectable img2img
initialization plus FLUX.2 multi-reference conditioning.
